In [65]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [66]:
# 1. 데이터 불러오기
df = pd.read_csv("청년정책목록_전체.csv")

In [67]:
# 2. 텍스트 결합 (결측값 처리 포함)
df['text'] = df['plcyExplnCn'].fillna('') + ' ' + \
             df['plcySprtCn'].fillna('') + ' ' + \
             df['plcyKywdNm'].fillna('')

In [68]:
# 3. 라벨 인코딩
le = LabelEncoder()
df['label'] = le.fit_transform(df['lclsfNm']) 

In [69]:
# 4. 텍스트 벡터화
vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df['text'])
y = df['label']

In [70]:
# 5. 학습/테스트 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [71]:
# 6. 모델 학습
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [72]:
# 7. 예측 및 성능 평가
y_pred = model.predict(X_test)


In [73]:
# 8. 지표 출력 (macro 평균 사용)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
acc = accuracy_score(y_test, y_pred)

# 9. 결과 출력
print(f"Precision (macro): {precision:.4f}")
print(f"Recall    (macro): {recall:.4f}")
print(f"F1-score  (macro): {f1:.4f}")
print(f"Accuracy         : {acc:.4f}")

Precision (macro): 0.8368
Recall    (macro): 0.7656
F1-score  (macro): 0.7919
Accuracy         : 0.8035


In [74]:
# 실제 예측

In [75]:
df_eval = pd.read_csv("청년정책목록_전처리완료_2025-06-09.csv")

In [76]:
# 2. 텍스트 결합
df_eval['text'] = df_eval['정책설명내용'].fillna('') + ' ' + \
                  df_eval['정책지원내용'].fillna('') + ' ' + \
                  df_eval['정책키워드명'].fillna('')

In [77]:
# 3. 벡터화 (학습된 vectorizer 사용)
X_eval = vectorizer.transform(df_eval['text'])

In [78]:
# 4. 실제 정답 라벨 (str → int)
y_true = le.transform(df_eval['정책대분류명'])

In [79]:
# 5. 예측
y_pred = model.predict(X_eval)

In [80]:
# 6. 성능 평가
print(classification_report(y_true, y_pred, target_names=le.classes_))
print("정확도:", accuracy_score(y_true, y_pred))

              precision    recall  f1-score   support

          기타       0.99      0.87      0.93       124
         일자리       0.91      0.99      0.95       173
          주거       1.00      1.00      1.00        30

    accuracy                           0.95       327
   macro avg       0.97      0.96      0.96       327
weighted avg       0.95      0.95      0.95       327

정확도: 0.9480122324159022


In [86]:
# 예측된 라벨 (숫자 → 텍스트) 생성
df['예측_대분류명'] = le.inverse_transform(model.predict(X))

In [87]:
print(df[['plcyNm', 'lclsfNm', '예측_대분류명']].head(10))

              plcyNm lclsfNm 예측_대분류명
0            주거안정장학금     일자리     일자리
1          부산 청년돌봄이음      기타      기타
2    서울 청년 마음건강 지원사업      기타     일자리
3          희망두배 청년통장      기타      기타
4             서울 영테크      기타      기타
5    부산청년 만원플러스 문화패스      기타      기타
6       부산 청년 소셜 다이닝      기타      기타
7       부산 청년활동 마일리지      기타      기타
8  부산 청년마음이음(일대일 상담)      기타      기타
9         경북 청년애꿈 수당     일자리     일자리


In [88]:
틀림 = df[df['lclsfNm'] != df['예측_대분류명']]
print(틀림[['plcyNo', 'plcyNm', 'lclsfNm', '예측_대분류명']].head())

           plcyNo                plcyNm lclsfNm 예측_대분류명
2    2.025050e+19       서울 청년 마음건강 지원사업      기타     일자리
53   2.025040e+19  「광명시 청년위원회」 4기 위원 모집      기타     일자리
54   2.025040e+19              육아나눔터 운영      기타     일자리
80   2.025040e+19       청년제대군인 진로탐색비 지원      기타     일자리
206  2.025020e+19          충북 창업인재 지원사업      기타     일자리
